# Python `defaultdict`

## Learning Objectives
- Understand the **problem** that `defaultdict` solves with regular `dict`
- Know how `defaultdict` uses a **default factory** to auto-create missing values
- Understand the `__missing__` hook that powers this behaviour
- Recognise the **common patterns** where `defaultdict` shines: grouping, counting, nested dicts
- Know the **gotchas**: phantom keys, serialisation differences, and when plain `dict` is better

## The Problem: `KeyError` with a Regular `dict`

A plain `dict` raises `KeyError` when you access a key that does not exist.
This forces you to check for the key first — or use `.get()` / `.setdefault()` — every time you want to accumulate or group values.

```python
d = {}
# KeyError
d['missing']
# KeyError — can't increment what doesn't exist
d['missing'] += 1
# KeyError — can't append to what doesn't exist
d['missing'].append(1)
```

The usual workarounds all add noise:

| Workaround | Code |
|---|---|
| `if key not in d` check | `if k not in d: d[k] = []; d[k].append(v)` |
| `.get()` with default | `d[k] = d.get(k, []) + [v]` (creates a new list every time) |
| `.setdefault()` | `d.setdefault(k, []).append(v)` — works but reads awkwardly |

In [1]:
# --- regular dict: three workarounds for the same grouping task ---

words = ['apple', 'ant', 'banana', 'avocado', 'blueberry', 'cherry', 'apricot']

# Workaround 1: explicit check
grouped_check = {}
for w in words:
    if w[0] not in grouped_check:
        grouped_check[w[0]] = []
    grouped_check[w[0]].append(w)

# Workaround 2: .get()
grouped_get = {}
for w in words:
    # creates a new list each iteration
    grouped_get[w[0]] = grouped_get.get(w[0], []) + [w]

# Workaround 3: .setdefault()
grouped_sd = {}
for w in words:
    grouped_sd.setdefault(w[0], []).append(w)

print("All three produce the same result:")
print(grouped_check)
print(grouped_get == grouped_check, grouped_sd == grouped_check)

All three produce the same result:
{'a': ['apple', 'ant', 'avocado', 'apricot'], 'b': ['banana', 'blueberry'], 'c': ['cherry']}
True True


## What is `defaultdict`?

`defaultdict` is a subclass of `dict` from the `collections` module.  
The only addition is a **default factory**: a zero-argument callable that is called automatically whenever a missing key is accessed.

```python
from collections import defaultdict

# factory = list
d = defaultdict(list)
# no KeyError — list() is called, [] is stored, then .append runs
d['missing'].append(1)
```

The factory is stored in `d.default_factory` and can be:

| Factory | Default value created |
|---|---|
| `list` | `[]` |
| `int` | `0` |
| `float` | `0.0` |
| `str` | `''` |
| `set` | `set()` |
| `dict` | `{}` |
| `lambda: <value>` | any custom default |
| `None` | disables auto-creation (same behaviour as regular `dict`) |

In [2]:
from collections import defaultdict

# Same grouping task — one clean line per word
grouped = defaultdict(list)
for w in words:
    grouped[w[0]].append(w)

print("Grouped by first letter:")
for letter, group in sorted(grouped.items()):
    print(f"  {letter}: {group}")

Grouped by first letter:
  a: ['apple', 'ant', 'avocado', 'apricot']
  b: ['banana', 'blueberry']
  c: ['cherry']


## How It Works: The `__missing__` Hook

`defaultdict` overrides the `__missing__` method that `dict` defines but leaves empty.

When you access `d[key]` and `key` is absent:
1. `dict.__getitem__` detects the miss
2. It calls `d.__missing__(key)`
3. `defaultdict.__missing__` calls `self.default_factory()` to get a new value
4. It **stores** that value under `key` in the dict
5. It **returns** the new value to the caller

This means the key is **permanently inserted** — it is not a transient default.  
Contrast this with `.get(key, default)`, which returns the default but **never stores it**.

In [3]:
from IPython.display import HTML, display

display(HTML("""
<figure style="margin:16px 0">
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 720 420" width="720" height="420"
     font-family="Segoe UI, Arial, sans-serif">
  <defs>
    <marker id="ma" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto">
      <path d="M0,0 L0,6 L9,3 z" fill="#555"/>
    </marker>
    <marker id="mr" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto">
      <path d="M0,0 L0,6 L9,3 z" fill="#c0392b"/>
    </marker>
    <marker id="mg" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto">
      <path d="M0,0 L0,6 L9,3 z" fill="#3a8c3f"/>
    </marker>
  </defs>

  <rect width="720" height="420" fill="#f9f9fb" rx="10"/>
  <text x="360" y="26" text-anchor="middle" font-size="13" font-weight="bold" fill="#1a1a2e">
    __missing__ call chain: d[key] vs d.get(key)
  </text>

  <!-- column divider -->
  <line x1="360" y1="38" x2="360" y2="408" stroke="#ddd" stroke-width="1.5" stroke-dasharray="5,4"/>

  <!-- ── column headers ── -->
  <rect x="18"  y="38" width="326" height="34" rx="7" fill="#dce9f7" stroke="#2166ac" stroke-width="2"/>
  <text x="181" y="60" text-anchor="middle" font-size="12" font-weight="bold" fill="#2166ac">defaultdict: d[key]</text>

  <rect x="376" y="38" width="326" height="34" rx="7" fill="#fde8e8" stroke="#c0392b" stroke-width="2"/>
  <text x="539" y="60" text-anchor="middle" font-size="12" font-weight="bold" fill="#c0392b">plain dict / both: d.get(key, default)</text>

  <!-- ── LEFT: step 1 — user call ── -->
  <rect x="68"  y="96" width="228" height="44" rx="7" fill="#dce9f7" stroke="#2166ac" stroke-width="1.8"/>
  <text x="182" y="115" text-anchor="middle" font-size="11" font-weight="bold" fill="#2166ac">user code</text>
  <text x="182" y="131" text-anchor="middle" font-size="10" fill="#2166ac">d[missing_key]</text>

  <line x1="182" y1="140" x2="182" y2="162" stroke="#555" stroke-width="1.8" marker-end="url(#ma)"/>
  <text x="192" y="155" font-size="9" fill="#555">triggers lookup</text>

  <!-- ── LEFT: step 2 — __getitem__ ── -->
  <rect x="68"  y="162" width="228" height="44" rx="7" fill="#e8ecf5" stroke="#777" stroke-width="1.5"/>
  <text x="182" y="181" text-anchor="middle" font-size="11" font-weight="bold" fill="#333">dict.__getitem__</text>
  <text x="182" y="197" text-anchor="middle" font-size="10" fill="#555">key absent — calls __missing__</text>

  <line x1="182" y1="206" x2="182" y2="228" stroke="#555" stroke-width="1.8" marker-end="url(#ma)"/>
  <text x="192" y="221" font-size="9" fill="#555">calls</text>

  <!-- ── LEFT: step 3 — __missing__ ── -->
  <rect x="68"  y="228" width="228" height="44" rx="7" fill="#fff3cd" stroke="#e6a817" stroke-width="2"/>
  <text x="182" y="247" text-anchor="middle" font-size="11" font-weight="bold" fill="#856404">d.__missing__(key)</text>
  <text x="182" y="263" text-anchor="middle" font-size="10" fill="#856404">only defaultdict has this</text>

  <line x1="182" y1="272" x2="182" y2="294" stroke="#555" stroke-width="1.8" marker-end="url(#ma)"/>
  <text x="192" y="287" font-size="9" fill="#555">invokes</text>

  <!-- ── LEFT: step 4 — factory ── -->
  <rect x="68"  y="294" width="228" height="44" rx="7" fill="#e8f5e9" stroke="#3a8c3f" stroke-width="1.8"/>
  <text x="182" y="313" text-anchor="middle" font-size="11" font-weight="bold" fill="#2e7d32">value = default_factory()</text>
  <text x="182" y="329" text-anchor="middle" font-size="10" fill="#2e7d32">zero-arg callable produces value</text>

  <line x1="182" y1="338" x2="182" y2="360" stroke="#3a8c3f" stroke-width="1.8" marker-end="url(#mg)"/>
  <text x="192" y="354" font-size="9" fill="#3a8c3f">stores + returns</text>

  <!-- ── LEFT: step 5 — stored ── -->
  <rect x="68"  y="360" width="228" height="44" rx="7" fill="#c8e6c9" stroke="#2e7d32" stroke-width="2"/>
  <text x="182" y="379" text-anchor="middle" font-size="11" font-weight="bold" fill="#1b5e20">d[key] = value  →  return value</text>
  <text x="182" y="395" text-anchor="middle" font-size="10" fill="#1b5e20">key permanently IN the dict</text>

  <!-- ── RIGHT: user call ── -->
  <rect x="426" y="96" width="228" height="44" rx="7" fill="#fde8e8" stroke="#c0392b" stroke-width="1.8"/>
  <text x="540" y="115" text-anchor="middle" font-size="11" font-weight="bold" fill="#c0392b">user code</text>
  <text x="540" y="131" text-anchor="middle" font-size="10" fill="#c0392b">d.get(missing_key, default)</text>

  <line x1="540" y1="140" x2="540" y2="162" stroke="#c0392b" stroke-width="1.8" marker-end="url(#mr)"/>

  <!-- ── RIGHT: dict.get impl ── -->
  <rect x="426" y="162" width="228" height="44" rx="7" fill="#e8ecf5" stroke="#777" stroke-width="1.5"/>
  <text x="540" y="181" text-anchor="middle" font-size="11" font-weight="bold" fill="#333">dict.get implementation</text>
  <text x="540" y="197" text-anchor="middle" font-size="10" fill="#555">key absent</text>

  <!-- long dashed bypass arrow — skips __missing__ entirely -->
  <line x1="540" y1="206" x2="540" y2="360" stroke="#c0392b" stroke-width="1.8"
        stroke-dasharray="6,3" marker-end="url(#mr)"/>

  <!-- "bypasses" label on right side of dashed line -->
  <rect x="556" y="258" width="90" height="36" rx="5" fill="#fff0f0" stroke="#c0392b" stroke-width="1"/>
  <text x="601" y="273" text-anchor="middle" font-size="9" fill="#c0392b">__missing__</text>
  <text x="601" y="286" text-anchor="middle" font-size="9" fill="#c0392b">NOT called</text>

  <!-- ── RIGHT: returned, not stored ── -->
  <rect x="426" y="360" width="228" height="44" rx="7" fill="#fde8e8" stroke="#c0392b" stroke-width="2"/>
  <text x="540" y="379" text-anchor="middle" font-size="11" font-weight="bold" fill="#c0392b">return default  (NOT stored)</text>
  <text x="540" y="395" text-anchor="middle" font-size="10" fill="#c0392b">key NOT in dict — no side-effect</text>
</svg>
<figcaption style="font-size:0.82em;color:#555;margin-top:6px;text-align:center">
  Left: <code>d[key]</code> on a <code>defaultdict</code> — inserts the key permanently via <code>__missing__</code>.<br>
  Right: <code>d.get(key, default)</code> — returns the default but never touches <code>__missing__</code> and never stores anything.
</figcaption>
</figure>
"""))

In [4]:
# Demonstrating that __missing__ stores the key
dd = defaultdict(list)

print("Keys before access :", list(dd.keys()))

# triggers __missing__ → list() → stored
val = dd['x']
# 'x' is now in the dict
print("Keys after dd['x'] :", list(dd.keys()))
print("Value of dd['x']   :", val)

print()

# .get() does NOT store the key
plain = {}
# returns [] but does NOT insert 'y'
result = plain.get('y', [])
# still empty
print("Keys after plain.get('y') :", list(plain.keys()))

print()

# You can call __missing__ explicitly to see what it does
dd2 = defaultdict(int)
returned = dd2.__missing__('z')
print(f"__missing__ returned: {returned!r}, 'z' in dd2: {'z' in dd2}, dd2['z']: {dd2['z']}")

Keys before access : []
Keys after dd['x'] : ['x']
Value of dd['x']   : []

Keys after plain.get('y') : []

__missing__ returned: 0, 'z' in dd2: True, dd2['z']: 0


## Key Differences: `dict` vs `defaultdict`

| Feature | `dict` | `defaultdict` |
|---|---|---|
| Missing key access `d[k]` | Raises `KeyError` | Calls `default_factory()`, stores & returns value |
| `.get(k, default)` | Returns default, **not stored** | Same — `.get()` does NOT trigger `__missing__` |
| `k in d` check | False for missing keys | False for missing keys (same) |
| Phantom key side-effect | No | Yes — `d[k]` creates the key even if you only wanted to read |
| `default_factory` attribute | Not present | Holds the callable (or `None`) |
| `repr` | `{'a': 1}` | `defaultdict(<class 'list'>, {'a': [1]})` |
| JSON serialisable directly | Yes | Yes (as a dict), but `default_factory` is lost |
| Subclass of `dict` | Is `dict` | Yes — passes `isinstance(d, dict)` |

The two are **identical** for all operations that do not involve a missing key.

In [5]:
import json

dd = defaultdict(list, a=[1, 2], b=[3])
plain = {'a': [1, 2], 'b': [3]}

print("isinstance checks:")
print(f"  isinstance(dd, dict)       : {isinstance(dd, dict)}")
print(f"  isinstance(dd, defaultdict): {isinstance(dd, defaultdict)}")
print()

print("repr:")
print(f"  dict       : {plain}")
print(f"  defaultdict: {dd}")
print()

print("JSON serialisation (defaultdict serialises as a plain dict):")
print(f"  {json.dumps(dd)}")
print()

print(".get() does NOT trigger __missing__ (no phantom key):")
dd2 = defaultdict(list)
result = dd2.get('missing', 'fallback')
print(f"  .get() returned: {result!r}, keys: {list(dd2.keys())}")
print()

# 'in' check also does NOT trigger __missing__
dd3 = defaultdict(list)
exists = 'missing' in dd3
print(f"  'missing' in dd3: {exists}, keys after: {list(dd3.keys())}")

isinstance checks:
  isinstance(dd, dict)       : True
  isinstance(dd, defaultdict): True

repr:
  dict       : {'a': [1, 2], 'b': [3]}
  defaultdict: defaultdict(<class 'list'>, {'a': [1, 2], 'b': [3]})

JSON serialisation (defaultdict serialises as a plain dict):
  {"a": [1, 2], "b": [3]}

.get() does NOT trigger __missing__ (no phantom key):
  .get() returned: 'fallback', keys: []

  'missing' in dd3: False, keys after: []


## Common Use Cases

### 1. Grouping items by a key

In [6]:
from collections import defaultdict

employees = [
    ('engineering', 'Alice'),
    ('marketing',   'Bob'),
    ('engineering', 'Carol'),
    ('marketing',   'Dave'),
    ('engineering', 'Eve'),
    ('hr',          'Frank'),
]

by_dept = defaultdict(list)
for dept, name in employees:
    by_dept[dept].append(name)

for dept, members in sorted(by_dept.items()):
    print(f"{dept:>12}: {members}")

 engineering: ['Alice', 'Carol', 'Eve']
          hr: ['Frank']
   marketing: ['Bob', 'Dave']


### 2. Counting with `int`

In [7]:
text = "the quick brown fox jumps over the lazy dog"

freq = defaultdict(int)
for ch in text:
    if ch != ' ':
        # int() → 0 on first access, then incremented
        freq[ch] += 1

# Top 5 most frequent characters
top5 = sorted(freq.items(), key=lambda x: -x[1])[:5]
print("Top 5 characters:", top5)

# Note: collections.Counter is more idiomatic for pure counting,
# but defaultdict(int) is useful when counting is mixed with other logic

Top 5 characters: [('o', 4), ('e', 3), ('t', 2), ('h', 2), ('u', 2)]


### 3. Nested `defaultdict` (tree-like structures)

In [8]:
# A nested defaultdict that auto-creates any depth
def nested_dd():
    return defaultdict(nested_dd)


tree = nested_dd()
tree['animals']['mammals']['dog'] = 'woof'
tree['animals']['mammals']['cat'] = 'meow'
tree['animals']['birds']['parrot'] = 'squawk'
tree['plants']['trees']['oak'] = 'deciduous'

print("Dog sound  :", tree['animals']['mammals']['dog'])
print("Parrot sound:", tree['animals']['birds']['parrot'])
print("Oak type   :", tree['plants']['trees']['oak'])
print()
print("Top-level keys:", list(tree.keys()))
print("Animal types  :", list(tree['animals'].keys()))

Dog sound  : woof
Parrot sound: squawk
Oak type   : deciduous

Top-level keys: ['animals', 'plants']
Animal types  : ['mammals', 'birds']


### 4. Accumulating unique items with `set`

In [9]:
# Track which users have visited each page (deduplicated)
visits = [
    ('home',    'alice'),
    ('about',   'bob'),
    # duplicate — should be ignored
    ('home',    'alice'),
    ('home',    'carol'),
    ('about',   'alice'),
    ('contact', 'bob'),
    ('home',    'bob'),
]

page_visitors = defaultdict(set)
for page, user in visits:
    page_visitors[page].add(user)

for page, visitors in sorted(page_visitors.items()):
    print(f"{page:>8}: {len(visitors)} unique visitors → {sorted(visitors)}")

   about: 2 unique visitors → ['alice', 'bob']
 contact: 1 unique visitors → ['bob']
    home: 3 unique visitors → ['alice', 'bob', 'carol']


### 5. Custom default value with `lambda`

In [10]:
# Default to a specific value that int/list/str can't express
# new players start with 100 points
scores = defaultdict(lambda: 100)

scores['alice'] += 50
scores['bob']   -= 20

# 150
print("Alice:", scores['alice'])
# 80
print("Bob  :", scores['bob'])
# 100 — default applied on first access
print("Carol:", scores['carol'])

print()
print("default_factory:", scores.default_factory)
print("All keys now:", dict(scores))

Alice: 150
Bob  : 80
Carol: 100

default_factory: <function <lambda> at 0x75528c998280>
All keys now: {'alice': 150, 'bob': 80, 'carol': 100}


## Gotchas

### 1. Phantom keys — reads create entries

The most common surprise: reading `d[k]` inserts `k` if it is missing.  
This silently inflates the dict and can hide bugs.

In [11]:
dd = defaultdict(list)
dd['a'].append(1)
dd['b'].append(2)

# Accidental read — perhaps a typo
# intended as a check, but 'c' is now in the dict
_ = dd['c']

print("Keys (c was never intentionally added):", list(dd.keys()))
print()

# Safe read: use .get() or 'in' — neither triggers __missing__
dd2 = defaultdict(list)
dd2['a'].append(1)

# returns None, no insertion
safe = dd2.get('c')
# False, no insertion
exists = 'c' in dd2
print(f".get('c'): {safe},  'c' in dd2: {exists},  keys: {list(dd2.keys())}")

Keys (c was never intentionally added): ['a', 'b', 'c']

.get('c'): None,  'c' in dd2: False,  keys: ['a']


### 2. Disabling the factory after construction

Set `default_factory = None` to turn off auto-creation once you are done building the dict.  
This converts it to dict-like read-only behaviour and surfaces bugs from unexpected key accesses.

In [12]:
# Build phase: factory active
inventory = defaultdict(int)
for item in ['apple', 'banana', 'apple', 'cherry', 'banana', 'apple']:
    inventory[item] += 1

print("Built inventory:", dict(inventory))

# Freeze: disable factory so accidental reads raise KeyError
inventory.default_factory = None

# works — key exists
print("apple count:", inventory['apple'])

try:
    # KeyError — factory is off
    _ = inventory['mango']
except KeyError as e:
    print(f"KeyError on 'mango': {e}")

Built inventory: {'apple': 3, 'banana': 2, 'cherry': 1}
apple count: 3
KeyError on 'mango': 'mango'


### 3. `defaultdict` in `repr` / serialisation

`defaultdict` serialises cleanly to JSON (as a plain dict), but the `default_factory` is lost.  
When round-tripping through JSON you get back a plain `dict`, not a `defaultdict`.

In [13]:
import json

dd = defaultdict(list, a=[1, 2], b=[3, 4])

serialised = json.dumps(dd)
print("JSON string:", serialised)

restored = json.loads(serialised)
# plain dict
print("Restored type:", type(restored).__name__)
# False
print("Has default_factory:", hasattr(restored, 'default_factory'))
print()

# If you need to preserve the factory, pickle round-trips correctly
import pickle
pickled  = pickle.dumps(dd)
from_pkl = pickle.loads(pickled)
# defaultdict
print("Pickle restored type:", type(from_pkl).__name__)
# <class 'list'>
print("Factory preserved   :", from_pkl.default_factory)

JSON string: {"a": [1, 2], "b": [3, 4]}
Restored type: dict
Has default_factory: False

Pickle restored type: defaultdict
Factory preserved   : <class 'list'>


## When to Use Which

| Situation | Prefer |
|---|---|
| Grouping items into lists | `defaultdict(list)` |
| Counting occurrences | `collections.Counter` (or `defaultdict(int)`) |
| Arbitrary deep nesting | `defaultdict(nested_dd)` with a recursive factory |
| Fixed set of known keys | Plain `dict` — missing key is a bug, not a feature |
| API/config dicts read by others | Plain `dict` — phantom key side-effect is surprising |
| One-off default with `.get()` | Plain `dict` — no need for a subclass |
| Read-heavy after build phase | Build with `defaultdict`, then freeze with `default_factory = None` |

**Rule of thumb:** reach for `defaultdict` when the pattern `d.setdefault(k, factory()).operation(v)` repeats throughout your code.  
If you only do it once or twice, `.setdefault()` on a plain `dict` is fine.

## Summary

| | `dict` | `defaultdict` |
|---|---|---|
| `d[missing]` | `KeyError` | calls `default_factory()`, stores result, returns it |
| `d.get(k)` | `None` (not stored) | `None` (not stored) — same |
| `k in d` | `False` | `False` — same |
| Phantom key risk | No | Yes — any `d[k]` read inserts `k` |
| Extra attribute | — | `default_factory` |
| Is a `dict` subclass | Itself | Yes |

`defaultdict` is `dict` plus a single hook (`__missing__`) that calls a factory on first access.  
It eliminates boilerplate for grouping and counting patterns, but the phantom-key side-effect means it is not a drop-in replacement everywhere — use a plain `dict` when an unexpected key should be an error.